# FASTA Sequences

### Import libraries and load dependencies:

In [1]:
import requests  # Library for sending HTTP requests to fetch data from URLs.
import pandas as pd  # Library for data manipulation and analysis.
from bs4 import BeautifulSoup  # Library for parsing HTML and XML documents.


### Define global variables

In [2]:
AMINO_ACIDS = {  # Dictionary mapping single-letter amino acid codes to their full names.
    'A': 'Ala',
    'C': 'Cys',
    'D': 'Asp',
    'E': 'Glu',
    'F': 'Phe',
    'G': 'Gly',
    'H': 'His',
    'I': 'Ile',
    'K': 'Lys',
    'L': 'Leu',
    'M': 'Met',
    'N': 'Asn',
    'P': 'Pro',
    'Q': 'Gln',
    'R': 'Arg',
    'S': 'Ser',
    'T': 'Thr',
    'V': 'Val',
    'W': 'Trp',
    'Y': 'Tyr',
}
AMINO_ACIDS_REVERSE = {value: key for key, value in AMINO_ACIDS.items()}  # Reverse dictionary for mapping full names to single-letter codes.


### Load datasets:

In [3]:
wild_df = pd.read_csv('../wild.csv', sep=';')  # Load wild-type dataset from CSV file.
variant_df = pd.read_csv('../variant.csv', sep=';')  # Load variant dataset from CSV file.

### Function to fetch wild FASTA sequence from MycoBrowser and save into a file

In [11]:
def get_fasta_sequence_wild(df):
    # Iterate through each row in the DataFrame.
    for index, row in df.iterrows():
        print(f"Saving FASTA sequence... ({index+1} from {df.shape[0]})")  # Log progress.
        gene = row["gene"]  # Extract gene name from the current row.
        url = row["mycobrowser_url"]  # Extract Mycobrowser URL.
        response = requests.get(url)  # Send GET request to fetch HTML content.
        if response.status_code == 200:  # Check if the request was successful.
            soup = BeautifulSoup(response.text, 'html.parser')  # Parse HTML content.
            fasta_section = soup.find_all('pre')  # Find all <pre> tags, which may contain FASTA sequences.
            if fasta_section:  # Check if any <pre> tags were found.
                for section in fasta_section:
                    if "Mycobacterium tuberculosis H37Rv|" in section.text:  # Ensure the section contains the correct FASTA sequence.
                        fasta_sequence = section.text.strip()  # Get the FASTA sequence and remove extra spaces.
                        fasta_sequence = fasta_sequence.split("\n")[1]  # Extract only the sequence part (ignoring the header).
                        df.loc[index, "fasta"] = fasta_sequence  # Add the sequence to the DataFrame.
            else:
                print(f"FASTA sequence not available")  # Log if no FASTA section was found.
        else:
            print(f"Gene information not found")  # Log if the request failed.
    return df  # Return the updated DataFrame with FASTA sequences added.


### Getting FASTA sequence from Mycobrowser url for all wild type

In [12]:
wild_df = get_fasta_sequence_wild(wild_df)  # Fetch and save FASTA sequences for complete wild types.

Saving FASTA sequence... (1 from 15)
Saving FASTA sequence... (2 from 15)
Saving FASTA sequence... (3 from 15)
Saving FASTA sequence... (4 from 15)
Saving FASTA sequence... (5 from 15)
Saving FASTA sequence... (6 from 15)
Saving FASTA sequence... (7 from 15)
Saving FASTA sequence... (8 from 15)
Saving FASTA sequence... (9 from 15)
Saving FASTA sequence... (10 from 15)
Saving FASTA sequence... (11 from 15)
Saving FASTA sequence... (12 from 15)
Saving FASTA sequence... (13 from 15)
Saving FASTA sequence... (14 from 15)
Saving FASTA sequence... (15 from 15)


In [15]:
# remove identifier column
wild_df = wild_df.drop(columns=['identifier'])  # Drop the 'identifier' column from the DataFrame.

In [16]:
wild_df.head()  # Display the first few rows of the updated DataFrame.

,gene,mycobrowser_url,fasta
0,atpE,https://mycobrowser.epfl.ch/genes/Rv1305,MDPTIAAGALIGGGLIMAGGAIGAGIGDGVAGNALISGVARQPEAQ...
1,Rv0678,https://mycobrowser.epfl.ch/genes/Rv0678,VSVNDGVDQMGAEPDIMEFVEQMGGYFESRSLTRLAGRLLGWLLVC...
2,tlyA,https://mycobrowser.epfl.ch/genes/Rv1694,VARRARVDAELVRRGLARSRQQAAELIGAGKVRIDGLPAVKPATAV...
3,ddn,https://mycobrowser.epfl.ch/genes/Rv3547,MPKSPPRFLNSPLSDFFIKWMSRINTWMYRRNDGEGLGGTFQKIPV...
4,embB,https://mycobrowser.epfl.ch/genes/Rv3795,MTQCASRRKSTPNRAILGAFASARGTRWVATIAGLIGFVLSVATPL...


In [18]:
print(wild_df[wild_df["fasta"].isnull()]) # Log genes without FASTA sequences due to their rRNA type. 

Empty DataFrame
Columns: [gene, mycobrowser_url, fasta]
Index: []


In [20]:
print(f"Dataset shape: {wild_df.shape}")  # Print the dataset's shape.
print(f"Rows: {wild_df.shape[0]}")  # Print the number of rows.
print(f"Columns: {wild_df.shape[1]}")  # Print the number of columns.


Dataset shape: (15, 3)
Rows: 15
Columns: 3


In [ ]:
wild_df.describe()  # Display summary statistics.

In [ ]:
wild_df.to_csv('../wild.csv', sep=';', index=False)  # Save the updated DataFrame to a new CSV file.

### Getting FASTA sequence for mutations

In [21]:
variant_df.head()  # Display the first few rows of the DataFrame.


,gene,identifier,variant
0,atpE,Rv1305,atpE_p.Ala63Pro
1,atpE,Rv1305,atpE_p.Asp28Ala
2,atpE,Rv1305,atpE_p.Asp28Gly
3,atpE,Rv1305,atpE_p.Asp28Val
4,atpE,Rv1305,atpE_p.Glu61Asp


In [22]:
print(f"Dataset shape: {variant_df.shape}")  # Print the dataset's shape.
print(f"Rows: {variant_df.shape[0]}")  # Print the number of rows.
print(f"Columns: {variant_df.shape[1]}")  # Print the number of columns.


Dataset shape: (384, 3)
Rows: 384
Columns: 3


In [23]:
variant_df.describe()  # Display summary statistics.


,gene,identifier,variant
count,384,384,384
unique,15,15,384
top,pncA,Rv2043c,atpE_p.Ala63Pro
freq,204,204,1


In [24]:
# Function to mutate a FASTA sequence based on a given mutation.
def get_fasta_mutation(fasta_selvagem, mutation):
    position = int(mutation[3:-3])  # Extract the position of the mutation.
    amino_acid = fasta_selvagem[position-1]  # Get the original amino acid at the position.
    if mutation[:3] != AMINO_ACIDS[amino_acid]:  # Check if the amino acid matches the expected value.
        print(f"Error converting fasta sequence")  # Log an error if the amino acid does not match.
        return "ERROR: amino-acid is not the same as expected"  # Return an error message.
    fasta_mutated = list(fasta_selvagem)  # Convert the FASTA sequence to a list for mutation.
    fasta_mutated[position-1] = AMINO_ACIDS_REVERSE[mutation[-3:]]  # Replace the amino acid at the position.
    fasta_mutated = ''.join(fasta_mutated)  # Convert the mutated list back to a string.
    return fasta_mutated  # Return the mutated FASTA sequence.


In [27]:
# Function to fetch and save FASTA sequences for variants.
def get_fasta_sequence_variant(df):
    for index, row in df.iterrows():
        print(f"Saving FASTA sequence... ({index+1} from {df.shape[0]})")  # Log progress.
        wild_row = wild_df[wild_df['gene'] == row['gene']]  # Find the corresponding wild type row.
        if wild_row.empty:  # Check if the wild type gene is not found.
            print(f"Gene {row['gene']} not found in wild_df")  # Log a warning.
            continue  # Skip to the next row.
        wild_fasta = wild_row['fasta'].values[0]  # Get the FASTA sequence for the wild type.
        variant = row['variant'].split('.')[1]  # Extract the mutation from the variant column.
        mutated_fasta = get_fasta_mutation(wild_fasta, variant)  # Generate the mutated FASTA sequence.
        df.at[index, 'fasta'] = mutated_fasta  # Add the mutated sequence to the DataFrame.
    return df  # Return the updated DataFrame with FASTA sequences added.

        

In [31]:
variant_df = get_fasta_sequence_variant(variant_df)# Fetch and save FASTA sequences for variants.
variant_df = variant_df.drop(columns=['identifier'])  # Drop the 'identifier' column from the DataFrame.

Saving FASTA sequence... (1 from 384)
Saving FASTA sequence... (2 from 384)
Saving FASTA sequence... (3 from 384)
Saving FASTA sequence... (4 from 384)
Saving FASTA sequence... (5 from 384)
Saving FASTA sequence... (6 from 384)
Saving FASTA sequence... (7 from 384)
Saving FASTA sequence... (8 from 384)
Saving FASTA sequence... (9 from 384)
Saving FASTA sequence... (10 from 384)
Saving FASTA sequence... (11 from 384)
Saving FASTA sequence... (12 from 384)
Saving FASTA sequence... (13 from 384)
Saving FASTA sequence... (14 from 384)
Saving FASTA sequence... (15 from 384)
Saving FASTA sequence... (16 from 384)
Saving FASTA sequence... (17 from 384)
Saving FASTA sequence... (18 from 384)
Saving FASTA sequence... (19 from 384)
Saving FASTA sequence... (20 from 384)
Saving FASTA sequence... (21 from 384)
Saving FASTA sequence... (22 from 384)
Saving FASTA sequence... (23 from 384)
Saving FASTA sequence... (24 from 384)
Saving FASTA sequence... (25 from 384)
Saving FASTA sequence... (26 from 

In [32]:
variant_df.head()  # Display the first few rows of the DataFrame.

,gene,variant,fasta
0,atpE,atpE_p.Ala63Pro,MDPTIAAGALIGGGLIMAGGAIGAGIGDGVAGNALISGVARQPEAQ...
1,atpE,atpE_p.Asp28Ala,MDPTIAAGALIGGGLIMAGGAIGAGIGAGVAGNALISGVARQPEAQ...
2,atpE,atpE_p.Asp28Gly,MDPTIAAGALIGGGLIMAGGAIGAGIGGGVAGNALISGVARQPEAQ...
3,atpE,atpE_p.Asp28Val,MDPTIAAGALIGGGLIMAGGAIGAGIGVGVAGNALISGVARQPEAQ...
4,atpE,atpE_p.Glu61Asp,MDPTIAAGALIGGGLIMAGGAIGAGIGDGVAGNALISGVARQPEAQ...


In [30]:
print(f"Dataset shape: {variant_df.shape}")  # Print the dataset's shape.
print(f"Rows: {variant_df.shape[0]}")  # Print the number of rows.
print(f"Columns: {variant_df.shape[1]}")  # Print the number of columns.

Dataset shape: (384, 4)
Rows: 384
Columns: 4


In [ ]:
variant_df.describe() # Display summary statistics.

In [ ]:
variant_df.to_csv(f"../variant.csv", index=False, sep=';') # Save the updated DataFrame with FASTA sequences to a CSV file.